___
# Motor de KPI
___
**DESCRIPCIÓN :**
Este notebook ejecuta las configuraciones obtenidas del notebook `Configuracion`.


In [0]:
%run ../Funciones

In [0]:
# Diccionario que define los esquemas y las tablas que serán cargadas (<esquema> : [<tablas>])
dict_load_tables = {
    "markpi": ["Dim_Escenario"]    
}

# Carga las tablas definidas en el diccionario dentro de DataFrames para su posterior procesamiento
dfs = load_tables(dict_load_tables)

In [0]:
# Lista de identificadores de widgets
widgets = ['Escenario', 'Dia']

# Crear widgets del notebook de Databricks
create_widgets(widgets)

In [0]:
param = get_widget_values(widgets)

In [0]:
dias = int(param["Dia"]) if param["Dia"] else 0

hoy = datetime.today().date()
fecha_corte = hoy - timedelta(days=dias)

datekey_corte = int(fecha_corte.strftime("%Y%m%d"))

In [0]:
param.update(get_dimensions_description(dfs, param))
nombre_escenario = param.get("ScenarioName", "default").lower()
id_escenario = int(param["Escenario"])

In [0]:
config_path = f"{bronze_env}.configuracion.configuracion_{nombre_escenario}"
print(f" └ {config_path =}")

In [0]:
dfs["config"] = (
    read_table(config_path)
    .select(
        "id_kpi",
        "id_escenario",
        "Id_Proceso",
        "Id_Equipo",
        "Consulta_Diaria",
        "Consulta_Turno"
    )
)

In [0]:
df_config = dfs["config"]

In [0]:
dfs_resultados = []
consultas_fallidas = []

for row in df_config.toLocalIterator():
    try:
        query = row.Consulta_Turno if row.Consulta_Turno not in (None, "") else row.Consulta_Diaria

        df_tmp = spark.sql(query)

        if row.Consulta_Turno:
            df_tmp = df_tmp.withColumnRenamed("Turno", "Id_Turno")
        else:
            df_tmp = df_tmp.withColumn("Id_Turno", F.lit(None).cast("int"))

        df_tmp = (
            df_tmp
            .withColumn("Id_Kpi", F.lit(row.id_kpi))
            .withColumn("Id_Escenario", F.lit(row.id_escenario))
        )

        dfs_resultados.append(df_tmp)

    except Exception as e:
        error_msg = str(e).split("]")[0] + "]"
    
        print(f"\n❌ Error en KPI {row.id_kpi}")
        print(f"   Error: {error_msg}")
    
        consultas_fallidas.append({
            "id_kpi": row.id_kpi,
            "Error": error_msg
        })

In [0]:
if not dfs_resultados:
    raise Exception("❌ Ninguna consulta se ejecutó correctamente")

if consultas_fallidas:
    print("\n==============================")
    print("KPIs QUE FALLARON")
    print("==============================")
    for c in consultas_fallidas:
        print(f"KPI {c['id_kpi']} -> {c['Error']}")
else:
    print("✔ Todas las consultas ejecutadas correctamente")

In [0]:
df_final = reduce(lambda a, b: a.unionByName(b), dfs_resultados)

In [0]:
df_final = df_final.select(
    F.col("id_kpi").alias("Id_Kpi"),
    F.col("DateKey").alias("Id_Tiempo"),
    F.col("id_escenario").alias("Id_Escenario"),
    F.col("Diario").alias("Valor_Kpi")
)

In [0]:
df_final = df_final.filter(
    F.col("Id_Tiempo") >= datekey_corte
)

In [0]:
df_final = df_final.withColumn(
    "Valor_Kpi",
    F.when(
        F.isnan("Valor_Kpi") |
        F.isnull("Valor_Kpi") |
        (F.col("Valor_Kpi") == float("inf")) |
        (F.col("Valor_Kpi") == float("-inf")),
        None
    ).otherwise(F.col("Valor_Kpi"))
)

In [0]:
table_name = f"{gold_env}.minajusta_operaciones.Fact_Kpi"

if not spark.catalog.tableExists(table_name) or spark.table(table_name).limit(1).count() == 0:
    
    (
        df_final
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )

else:

    spark.sql(f"""
        DELETE FROM {table_name}
        WHERE Id_Tiempo >= {datekey_corte}
            AND Id_Escenario = {id_escenario}
    """)

    (
        df_final
        .write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(table_name)
    )